In [1]:
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
import vector
import awkward as ak
import os
import glob

In [2]:
def load_parquet(input_path: str, columns: list = None) -> ak.Array:
    """ Loads the contents of the .parquet file specified by the input_path
    Args:
        input_path : str
            The path to the .parquet file to be loaded.
        columns : list
            Names of the columns/branches to be loaded from the .parquet file
    Returns:
        input_data : ak.Array
            The data from the .parquet file
    """
    ret = ak.from_parquet(input_path, columns=columns)
    ret = ak.Array({k: ret[k] for k in ret.fields})
    return ret

In [3]:
def load_all_data(input_loc: str, n_files: int = None, columns: list = None) -> ak.Array:
    """Loads all .parquet files specified by the input. The input can be a list of input_paths, a directory where the files
    are located or a wildcard path.
    Args:
        input_loc : str
            Location of the .parquet files.
        n_files : int
            [default: None] Maximum number of input files to be loaded. By default all will be loaded.
        columns : list
            [default: None] Names of the columns/branches to be loaded from the .parquet file. By default all columns will
            be loaded
    Returns:
        input_data : ak.Array
            The concatenated data from all the loaded files
    """
    if n_files == -1:
        n_files = None
    if isinstance(input_loc, list):
        input_files = input_loc[:n_files]
    elif isinstance(input_loc, str):
        if os.path.isdir(input_loc):
            input_loc = os.path.expandvars(input_loc)
            input_files = glob.glob(os.path.join(input_loc, "*.parquet"))[:n_files]
        elif "*" in input_loc:
            input_files = glob.glob(input_loc)[:n_files]
        else:
            raise ValueError(f"Unexpected input_loc")
    else:
        raise ValueError(f"Unexpected input_loc")
    input_data = []
    for i, file_path in enumerate(input_files):
        print(f"[{i+1}/{len(input_files)}] Loading from {file_path}")
        try:
            input_data.append(load_parquet(file_path, columns=columns))
        except ValueError:
            print(f"{file_path} does not exist")
    if len(input_data) > 0:
        data = ak.concatenate(input_data)
        print("Input data loaded")
    else:
        raise ValueError(f"No files found in {input_loc}")
    return data

In [4]:
ggf = load_all_data("/home/norman/vbf-tagger/vbf_tagger/data/22pre/hh_ggf/preprocessed_filtered/train/", 3)

[1/3] Loading from /home/norman/vbf-tagger/vbf_tagger/data/22pre/hh_ggf/preprocessed_filtered/train/hh_ggf_hbb_htt_kl5_kt1_powheg_events_0.parquet
[2/3] Loading from /home/norman/vbf-tagger/vbf_tagger/data/22pre/hh_ggf/preprocessed_filtered/train/hh_ggf_hbb_htt_kl2p45_kt1_powheg_events_1.parquet
[3/3] Loading from /home/norman/vbf-tagger/vbf_tagger/data/22pre/hh_ggf/preprocessed_filtered/train/hh_ggf_hbb_htt_kl0_kt1_powheg_events_0.parquet
Input data loaded


In [41]:
ttb = load_all_data("/home/norman/vbf-tagger/vbf_tagger/data/22pre/tt_sl/preprocessed_filtered/train/", 3)

[1/3] Loading from /home/norman/vbf-tagger/vbf_tagger/data/22pre/tt_sl/preprocessed_filtered/train/events_131.parquet
[2/3] Loading from /home/norman/vbf-tagger/vbf_tagger/data/22pre/tt_sl/preprocessed_filtered/train/events_295.parquet
[3/3] Loading from /home/norman/vbf-tagger/vbf_tagger/data/22pre/tt_sl/preprocessed_filtered/train/events_146.parquet
Input data loaded


In [42]:
ttb.VBFJet

<Array [[], [], [], [], ..., [], [], [], []] type='1896 * var * {btagDeepFl...'>

In [44]:
ttb.TrainingJet.isVBF

<Array [[1, 0, 0, 0], [0, ..., 1], ..., [0, 0, 0]] type='1896 * var * ?float64'>

In [8]:
def initialize_p4(data):
    return ak.zip(
            {
                "pt": data.pt,
                "eta": data.eta,
                "phi": data.phi,
                "mass": data.mass,
            },
            with_name="Momentum4D"
        )

In [9]:
jets0 = initialize_p4(ggf.TrainingJet)

In [10]:
mask_valid = ak.num(jets0) > 3
isVBF = ggf.TrainingJet.isVBF[mask_valid]

In [15]:
jets = jets0[mask_valid]
jets_novec = ggf.TrainingJet[mask_valid]
pairs_novec = ak.combinations(jets_novec, 2, fields=["j1", "j2"])
pairs = ak.combinations(jets, 2, fields=["j1", "j2"])
pairs_isVBF = ak.combinations(isVBF, 2, fields=["j1", "j2"])

In [29]:
pairs[0]

<Array [{j1: {...}, j2: {...}}, ..., {...}] type='6 * {j1: Momentum4D[pt: f...'>

In [30]:
pairs_isVBF[0]

<Array [{j1: 0, j2: 0}, ..., {j1: 0, ...}] type='6 * {j1: ?float64, j2: ?fl...'>

In [35]:
masktest = (pairs_isVBF.j1==1) & (pairs_isVBF.j2==1)

In [39]:
pairs_isVBF[masktest]

<Array [[], [], [], [], ..., [], [], [], []] type='45550 * var * ?{j1: ?flo...'>